# El Clásico case study — development out-of-fold predictions

We will revisit two matches chosen in advance: Real Madrid 0–4 Barcelona (21 November 2015) and Barcelona 1–2 Real Madrid (2 April 2016). The case study uses the frozen Phase 8 candidate's saved outer-fold out-of-fold (OOF) probabilities for these shots. It does not call the final model to predict its own training rows or refit any model.

Each case match was excluded from its outer fold's base-model and calibration fitting. These are still development matches, and the candidate was selected using overall development results. This is an explanatory OOF case study, not an independent test evaluation or a pre-match forecast.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in (cwd, *cwd.parents) if (p / '.venv').is_dir() and (p / 'README.md').is_file()), None)
if project_root is None:
    raise RuntimeError('Could not find the project root containing .venv/ and README.md. Open this notebook from the project checkout.')
if Path(sys.prefix).resolve() != (project_root / '.venv').resolve():
    raise RuntimeError(f'Select the project .venv kernel in VS Code. Active environment: {sys.prefix}')
if not (project_root / '.venv' / 'Scripts' / 'python.exe').exists():
    raise RuntimeError('The project .venv Python executable is missing.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f'Project: {project_root}')
print(f'Kernel:  {sys.executable}')

## 1. Verify saved sources and run the case study

The script verifies the Phase 8 OOF table and selection, the locked Phase 6 fold map, and the Phase 4 shot, reconciliation, and own-goal CSV hashes. It joins the saved predictions to source shots by event ID and rejects a fold, match, or label mismatch. The protocol is frozen before it simulates the two matches.

In [ ]:
import subprocess

script = project_root / 'scripts' / 'el_clasico_case_study.py'
result = subprocess.run([sys.executable, str(script)], cwd=project_root, check=False, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'El Clásico case study failed with exit code {result.returncode}; see the output above.')

## 2. Inspect match and team summaries

For each side, the expected shot goals equal the sum of its saved OOF probabilities. The exact outcome probabilities come from convolving every observed shot's Bernoulli distribution. The Monte Carlo comparison checks only finite-simulation variation for these fixed probabilities.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

tables = project_root / 'reports' / 'tables'
summary = pd.read_csv(tables / 'el_clasico_match_summary.csv', float_precision='round_trip')
report = json.loads((project_root / 'docs' / 'el_clasico_report.json').read_text(encoding='utf-8'))
shots = pd.read_csv(tables / 'el_clasico_shots.csv', float_precision='round_trip')
print(f"Scope: {report['scope']} | candidate: {report['candidate']} | trials per match: {report['trial_count_per_match']:,}")
print('Per-match OOF fold and official-score audit:')
display(summary[['match_date', 'home_team_name', 'away_team_name', 'outer_oof_fold', 'official_home_goals', 'official_away_goals', 'observed_shot_goals_home', 'observed_shot_goals_away', 'own_goals_credited_home', 'own_goals_credited_away']])
teams = []
for row in summary.itertuples(index=False):
    teams.extend([
        {'match_id': row.match_id, 'team': row.home_team_name, 'shots': row.home_shot_count, 'expected_goals': row.home_expected_shot_goals, 'MC_mean_goals': row.home_mc_mean_goals, 'MCSE': row.home_mean_mcse, 'expected_points': row.home_expected_points_exact},
        {'match_id': row.match_id, 'team': row.away_team_name, 'shots': row.away_shot_count, 'expected_goals': row.away_expected_shot_goals, 'MC_mean_goals': row.away_mc_mean_goals, 'MCSE': row.away_mean_mcse, 'expected_points': row.away_expected_points_exact},
    ])
display(pd.DataFrame(teams))
outcomes = []
for row in summary.itertuples(index=False):
    for name, exact, mc in [('Home win', row.exact_home_win_probability, row.mc_home_win_probability), ('Draw', row.exact_draw_probability, row.mc_draw_probability), ('Away win', row.exact_away_win_probability, row.mc_away_win_probability)]:
        outcomes.append({'match_id': row.match_id, 'outcome': name, 'exact_probability': exact, 'MC_probability': mc, 'absolute_difference': abs(exact - mc)})
display(pd.DataFrame(outcomes))
print('Source OOF rows, including event identifiers and fold assignments: reports/tables/el_clasico_shots.csv')
print(shots.groupby(['match_id', 'fold']).size().rename('case_shot_rows'))
for match_id in summary.match_id:
    filename = f'el_clasico_{int(match_id)}_outcomes.png'
    display(Image(filename=str(project_root / 'reports' / 'figures' / filename)))

## 3. View the full joint scoreline distributions

Each heatmap cell is the exact joint probability for that home-away score. The exported table preserves the full mathematical support, including scorelines that received zero Monte Carlo draws. The bar charts compare exact home/draw/away probabilities with the seeded simulation.

In [ ]:
scorelines = pd.read_csv(tables / 'el_clasico_scorelines.csv', float_precision='round_trip')
for match_id in summary.match_id:
    top = scorelines.loc[scorelines.match_id == match_id].sort_values('exact_probability', ascending=False, kind='stable').head(5)
    print(f'Top five scorelines for match {int(match_id)}; full scoreline CSV retains every support cell:')
    display(top[['home_goals', 'away_goals', 'exact_probability', 'mc_count', 'mc_probability']])
display(Image(filename=str(project_root / 'reports' / 'figures' / 'el_clasico_joint_scorelines.png')))

## 4. Interpret this as a development case study

The outer validation fold excluded each match from that fold's base-model and sigmoid-calibration fitting, so the OOF probabilities are preferable here to full-training resubstitution predictions. However, Phase 8 selected the global candidate using development results that include all folds. This case study therefore does not provide a fresh or independent test estimate.

The simulation is conditional on the shots that happened and treats their goal outcomes as independent Bernoulli trials. It does not model whether a rebound shot would have occurred after a different earlier outcome, and it does not simulate own goals. Official scores, ordinary shot goals, and own-goal credits remain separate in the summary. Use the Phase 9 test-set simulation for the held-out evaluation; use this notebook to study these two historically interesting matches.

Full provenance is in `docs/el_clasico_protocol.json` and `docs/el_clasico_report.json`.